# Step 5: Pilot Run (Round 1 — Memory Form)

这份 notebook 执行 Round 1 pilot run：在 10 个 `2WikiMultiHopQA` target 上比较三个条件：

- `no_memory`
- `episodic_trace`
- `cross_episode_consolidation`

每个 target task 分别在 `relevant` 和 `irrelevant` split 上跑 memory 条件，
因此每个 target 最多产生 5 条 run 记录（1 no_memory + 2 relevant + 2 irrelevant）。

### 路径策略

与 `04_artifact_generation.ipynb` 一致：
- 优先读 `PROJECT_ROOT_OVERRIDE`
- 其次读环境变量 `SELECT_TRANSFER_ROOT`
- 否则自动探测

### 模型

默认使用云端已下载的本地 `Qwen/Qwen3.5-9B`（与 artifact generation 同一模型）。
也可以切换到 API backend（设 `USE_API = True`）。

### 云端最小文件依赖

```
2026_SelectTransfer/
├── notebooks/
│   └── 05_pilot_run.ipynb          # 本文件
├── pilot/
│   └── archive/
│       ├── taxonomy_round1.csv
│       ├── source_sets_round1.csv
│       └── pairing_table_round1.csv
├── results/
│   ├── 01_sampling/
│   │   └── sampled_20_full.json
│   └── 02_hotpotqa_comparison_expansion/
│       └── candidate_batch_filtered_full.json
├── artifacts/
│   ├── hp_bridge_set_01/
│   │   ├── episodic_trace.md
│   │   └── cross_episode_consolidation.md
│   └── hp_comparison_set_01/
│       ├── episodic_trace.md
│       └── cross_episode_consolidation.md
└── results/
    └── 04_pilot_run/              # 输出目录（自动创建）
```

In [1]:
import csv
import gc
import json
import os
import re
import string
import time
from pathlib import Path

try:
    import torch
except ImportError:
    torch = None

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
except ImportError:
    AutoModelForCausalLM = None
    AutoTokenizer = None

## 1. 路径检测与配置

In [2]:
# ── 修改这里来适配你的云端路径 ──
PROJECT_ROOT_OVERRIDE = '/root/2026_SelectTransfer'


def detect_project_root():
    candidates = []
    if PROJECT_ROOT_OVERRIDE.strip():
        candidates.append(Path(PROJECT_ROOT_OVERRIDE).expanduser())
    env_root = os.environ.get('SELECT_TRANSFER_ROOT', '').strip()
    if env_root:
        candidates.append(Path(env_root).expanduser())
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd, cwd.parent,
        cwd / '2026_SelectTransfer',
        Path('/content/2026_SelectTransfer'),
        Path('/workspace/2026_SelectTransfer'),
        Path('/root/2026_SelectTransfer'),
        Path('/kaggle/working/2026_SelectTransfer'),
    ])
    seen = set()
    checked = []
    for c in candidates:
        if c in seen:
            continue
        seen.add(c)
        checked.append(str(c))
        if (
            (c / 'pilot' / 'archive' / 'pairing_table_round1.csv').exists()
            and (c / 'artifacts' / 'hp_bridge_set_01' / 'episodic_trace.md').exists()
        ):
            return c
    raise FileNotFoundError(
        'Could not locate project root. '
        'Set PROJECT_ROOT_OVERRIDE or SELECT_TRANSFER_ROOT. '
        'Checked: ' + ' | '.join(checked)
    )


PROJECT_ROOT = detect_project_root()
ARCHIVE_DIR = PROJECT_ROOT / 'pilot' / 'archive'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
RESULTS_DIR = PROJECT_ROOT / 'results' / '04_pilot_run'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RESULTS_DIR  =', RESULTS_DIR)

PROJECT_ROOT = /root/2026_SelectTransfer
RESULTS_DIR  = /root/2026_SelectTransfer/results/04_pilot_run


In [3]:
# ── 模型与运行配置 ──
MODEL_ID = 'Qwen/Qwen3.5-9B'
FALLBACK_MODEL_ID = 'Qwen/Qwen3.5-4B'
HF_TOKEN = os.environ.get('HF_TOKEN', '')

# 设为 True 才会真正调用模型；False = dry run（只构造 prompt，不生成）
RUN_GENERATION = True

# 设为 True 使用外部 API（需要 API_KEY）；False = 本地 transformers 推理
USE_API = False
API_KEY = os.environ.get('OPENAI_API_KEY', '')
API_BASE_URL = os.environ.get('OPENAI_API_BASE', '')
API_MODEL = os.environ.get('PILOT_API_MODEL', 'gpt-4o-mini')

ENABLE_THINKING = False
MAX_NEW_TOKENS = 512
DO_SAMPLE = False

print('MODEL_ID =', MODEL_ID)
print('RUN_GENERATION =', RUN_GENERATION)
print('USE_API =', USE_API)
print('torch available =', torch is not None)
if torch is not None and torch.cuda.is_available():
    print('CUDA device =', torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print('CUDA memory (GB) =', round(total_gb, 2))

MODEL_ID = Qwen/Qwen3.5-9B
RUN_GENERATION = True
USE_API = False
torch available = True
CUDA device = NVIDIA GeForce RTX 4090
CUDA memory (GB) = 47.37


## 2. 读取冻结输入

In [4]:
def read_csv(path):
    with path.open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))


def load_json(path):
    return json.loads(path.read_text(encoding='utf-8'))


# ── Pairing table ──
pairing_rows = read_csv(ARCHIVE_DIR / 'pairing_table_round1.csv')
print(f'pairing rows: {len(pairing_rows)}')

# ── Taxonomy (for gold answers) ──
taxonomy_rows = read_csv(ARCHIVE_DIR / 'taxonomy_round1.csv')
taxonomy_map = {row['task_id']: row for row in taxonomy_rows}
print(f'taxonomy rows: {len(taxonomy_rows)}')

# ── Task payloads (for context) ──
sampled_rows = load_json(PROJECT_ROOT / 'results' / '01_sampling' / 'sampled_20_full.json')
expanded_rows = load_json(
    PROJECT_ROOT / 'results' / '02_hotpotqa_comparison_expansion' / 'candidate_batch_filtered_full.json'
)
all_payload = {row['task_id']: row for row in sampled_rows + expanded_rows}
print(f'task payloads: {len(all_payload)}')

# ── Verify all target tasks have payloads ──
target_ids = [r['target_task_id'] for r in pairing_rows]
missing = [tid for tid in target_ids if tid not in all_payload]
print(f'target tasks: {len(target_ids)}, missing payloads: {missing}')

pairing rows: 10
taxonomy rows: 35
task payloads: 35
target tasks: 10, missing payloads: []


## 3. 读取 artifacts

In [5]:
def load_artifact(source_set_id, artifact_type):
    """Load artifact markdown content. artifact_type: 'episodic_trace' or 'cross_episode_consolidation'"""
    path = ARTIFACTS_DIR / source_set_id / f'{artifact_type}.md'
    if not path.exists():
        raise FileNotFoundError(f'Artifact not found: {path}')
    return path.read_text(encoding='utf-8').strip()


# Pre-load all artifacts
artifact_cache = {}
for source_set_id in ['hp_bridge_set_01', 'hp_comparison_set_01']:
    for atype in ['episodic_trace', 'cross_episode_consolidation']:
        key = (source_set_id, atype)
        artifact_cache[key] = load_artifact(source_set_id, atype)
        print(f'{source_set_id}/{atype}: {len(artifact_cache[key])} chars')

print(f'\nartifact cache: {len(artifact_cache)} entries')

hp_bridge_set_01/episodic_trace: 3161 chars
hp_bridge_set_01/cross_episode_consolidation: 3100 chars
hp_comparison_set_01/episodic_trace: 3787 chars
hp_comparison_set_01/cross_episode_consolidation: 3290 chars

artifact cache: 4 entries


## 4. Prompt 构造

严格遵循 `protocol/pilot-prompt-scaffold.md` 的定义。

In [6]:
def build_context_paragraphs(raw_context):
    """
    raw_context: JSON string -> list of [title, [sent1, sent2, ...]]
    Returns formatted markdown paragraphs.
    """
    if isinstance(raw_context, str):
        raw_context = json.loads(raw_context)
    paragraphs = []
    for entry in raw_context:
        title = entry[0]
        sentences = entry[1] if len(entry) > 1 else []
        text = ' '.join(str(s) for s in sentences)
        paragraphs.append(f'### {title}\n{text}')
    return '\n\n'.join(paragraphs)


def assemble_prompt(target_task, condition, artifact_content=None):
    """
    Build the full prompt for one run.
    condition: 'no_memory' | 'episodic_trace' | 'cross_episode_consolidation'
    """
    raw = target_task['raw']
    context = build_context_paragraphs(raw['context'])
    question = target_task['question']

    base_header = (
        'You are a question-answering agent. '
        'Your task is to answer a multi-hop reasoning question '
        'using the provided context paragraphs.'
    )

    context_section = f'## Context\n\n{context}'
    question_section = f'## Question\n\n{question}'

    if condition == 'no_memory':
        memory_section = ''
    else:
        memory_section = (
            '## Past Experience\n\n'
            'The following notes summarize patterns from previously solved tasks '
            'that may or may not be relevant to the current question. '
            'Use them only if they help your reasoning — do not force-apply them.\n\n'
            f'{artifact_content}'
        )

    instructions_section = (
        '## Instructions\n\n'
        '- Read all context paragraphs carefully.\n'
        '- Identify the reasoning chain needed to answer the question.\n'
        '- Provide your final answer as a short phrase (not a full sentence).\n'
        '- If the question asks "which", "who", or "what", '
        'respond with the specific entity name.\n'
        '- If the question asks for a comparison, '
        'respond with the entity that satisfies the comparison.'
    )

    answer_section = '## Answer'

    parts = [base_header, '', context_section, '', question_section]
    if memory_section:
        parts += ['', memory_section]
    parts += ['', instructions_section, '', answer_section]

    return '\n\n'.join(parts)

## 5. 评分函数

In [7]:
def normalize_answer(text):
    """Lowercase, strip articles/punctuation, collapse whitespace."""
    text = text.lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = ' '.join(text.split())
    return text


def compute_em(pred, gold):
    return int(normalize_answer(pred) == normalize_answer(gold))


def compute_f1(pred, gold):
    pred_tokens = set(normalize_answer(pred).split())
    gold_tokens = set(normalize_answer(gold).split())
    if not pred_tokens or not gold_tokens:
        return 0.0
    common = pred_tokens & gold_tokens
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def extract_answer(model_output):
    """Extract the answer after '## Answer' marker."""
    if '## Answer' in model_output:
        return model_output.split('## Answer')[-1].strip()
    lines = [l.strip() for l in model_output.strip().split('\n') if l.strip()]
    return lines[-1] if lines else ''


# Quick sanity check
assert compute_em('Jean-Baptiste Le Prince', 'Jean-Baptiste Le Prince') == 1
assert compute_em('the Jean-Baptiste Le Prince', 'Jean-Baptiste Le Prince') == 1
assert compute_f1('Jean-Baptiste Le Prince', 'Jean-Baptiste Le Prince') == 1.0
print('scoring functions OK')

scoring functions OK


## 6. 构造 run plan

每个 target task 产生以下 runs：

| run | condition | split | source_set_id |
|---|---|---|---|
| 1 | `no_memory` | `relevant` | (relevant source set，仅用于 traceability) |
| 2 | `episodic_trace` | `relevant` | relevant source set |
| 3 | `cross_episode_consolidation` | `relevant` | relevant source set |
| 4 | `episodic_trace` | `irrelevant` | irrelevant source set |
| 5 | `cross_episode_consolidation` | `irrelevant` | irrelevant source set |

`no_memory` 只需跑一次（与 split 无关，因为没有 memory 注入），
但在结果表中同时出现在两个 split 下以便对比。

In [8]:
run_plan = []

for pair in pairing_rows:
    tid = pair['target_task_id']
    rel_ssid = pair['relevant_source_set_id']
    irr_ssid = pair['irrelevant_source_set_id']

    # no_memory: 同一个 prompt，结果复制到两个 split
    run_plan.append({
        'run_id': f'r1_no_memory_{tid}_relevant',
        'target_task_id': tid,
        'split': 'relevant',
        'condition': 'no_memory',
        'source_set_id': rel_ssid,
        'artifact_type': None,
    })
    run_plan.append({
        'run_id': f'r1_no_memory_{tid}_irrelevant',
        'target_task_id': tid,
        'split': 'irrelevant',
        'condition': 'no_memory',
        'source_set_id': irr_ssid,
        'artifact_type': None,
    })

    # relevant split: episodic + consolidation
    for atype in ['episodic_trace', 'cross_episode_consolidation']:
        run_plan.append({
            'run_id': f'r1_{atype}_{tid}_relevant',
            'target_task_id': tid,
            'split': 'relevant',
            'condition': atype,
            'source_set_id': rel_ssid,
            'artifact_type': atype,
        })

    # irrelevant split: episodic + consolidation
    for atype in ['episodic_trace', 'cross_episode_consolidation']:
        run_plan.append({
            'run_id': f'r1_{atype}_{tid}_irrelevant',
            'target_task_id': tid,
            'split': 'irrelevant',
            'condition': atype,
            'source_set_id': irr_ssid,
            'artifact_type': atype,
        })

print(f'total runs planned: {len(run_plan)}')
print(f'  no_memory:                    {sum(1 for r in run_plan if r["condition"] == "no_memory")}')
print(f'  episodic_trace:               {sum(1 for r in run_plan if r["condition"] == "episodic_trace")}')
print(f'  cross_episode_consolidation:  {sum(1 for r in run_plan if r["condition"] == "cross_episode_consolidation")}')
print(f'  relevant split:               {sum(1 for r in run_plan if r["split"] == "relevant")}')
print(f'  irrelevant split:             {sum(1 for r in run_plan if r["split"] == "irrelevant")}')

total runs planned: 60
  no_memory:                    20
  episodic_trace:               20
  cross_episode_consolidation:  20
  relevant split:               30
  irrelevant split:             30


## 7. Prompt Preview（dry run 检查）

在真正调用模型之前，先检查一两个 prompt 长什么样。

In [9]:
# Preview: first target, no_memory vs episodic_trace
sample_run = run_plan[0]
sample_task = all_payload[sample_run['target_task_id']]

prompt_no_mem = assemble_prompt(sample_task, 'no_memory')
print('=== no_memory prompt ===')
print(f'chars: {len(prompt_no_mem)}')
print(prompt_no_mem[:1500])
print('\n... (truncated) ...\n')

# episodic_trace relevant
sample_run_ep = [r for r in run_plan if r['condition'] == 'episodic_trace' and r['split'] == 'relevant'][0]
art_content = artifact_cache[(sample_run_ep['source_set_id'], 'episodic_trace')]
prompt_ep = assemble_prompt(
    all_payload[sample_run_ep['target_task_id']],
    'episodic_trace',
    artifact_content=art_content,
)
print('=== episodic_trace (relevant) prompt ===')
print(f'chars: {len(prompt_ep)}')
print(prompt_ep[:2000])
print('\n... (truncated) ...')

=== no_memory prompt ===
chars: 2812
You are a question-answering agent. Your task is to answer a multi-hop reasoning question using the provided context paragraphs.



## Context

### Billy Magoulias
Billy Magoulias( born 23 January 1997) is a Greek international rugby league footballer who plays as a for the Cronulla- Sutherland Sharks in the NRL.

### William Pratt
William or Billy Pratt may refer to:

### Bill Phillips
Bill or Billy Phillips may refer to:

### Jean-Baptiste Le Prince
Jean- Baptiste Le Prince( September 17, 1734 – September 30, 1781) was an important French etcher and painter. Le Prince first studied painting techniques in his native Metz. He then travelled to Paris around 1750 and became a leading student of the great painter, François Boucher( 1703 – 1770). Le Prince's early paintings in both theme and style are comparable to his master's rococo techniques. In 1758 Le Prince journeyed to Russia to work for Catherine the Great at the Imperial Palace, St. Petersburg

## 8. 模型加载（本地 transformers）

In [10]:
def infer_torch_dtype():
    if torch is None:
        raise ImportError('torch is not available.')
    if torch.cuda.is_available():
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.float32


def load_local_model(model_id):
    if torch is None or AutoTokenizer is None:
        raise ImportError('torch or transformers not available.')
    token = HF_TOKEN or None
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map='auto',
        torch_dtype=infer_torch_dtype(),
        token=token,
    )
    model.eval()
    return tokenizer, model


tokenizer = None
model = None

if RUN_GENERATION and not USE_API:
    tokenizer, model = load_local_model(MODEL_ID)
    print(f'Loaded local model: {MODEL_ID}')
elif RUN_GENERATION and USE_API:
    print(f'Will use API backend: {API_MODEL}')
else:
    print('RUN_GENERATION = False -> dry run only')

`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Loaded local model: Qwen/Qwen3.5-9B


## 9. Generation 函数

In [11]:
def generate_local(prompt_text):
    """Generate answer using local transformers model."""
    messages = [
        {'role': 'user', 'content': prompt_text},
    ]
    try:
        chat_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=ENABLE_THINKING,
        )
    except TypeError:
        chat_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )

    inputs = tokenizer([chat_text], return_tensors='pt')
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    prompt_tokens = inputs['input_ids'].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_ids = output_ids[0][prompt_tokens:]
    completion_tokens = len(new_ids)
    text = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    if not ENABLE_THINKING:
        text = text.replace('<think>', '').replace('</think>', '').strip()
    return text, prompt_tokens + completion_tokens


def generate_api(prompt_text):
    """Generate answer using OpenAI-compatible API."""
    import httpx
    headers = {'Authorization': f'Bearer {API_KEY}', 'Content-Type': 'application/json'}
    payload = {
        'model': API_MODEL,
        'messages': [{'role': 'user', 'content': prompt_text}],
        'max_tokens': MAX_NEW_TOKENS,
        'temperature': 0.0,
    }
    base = API_BASE_URL.rstrip('/') if API_BASE_URL else 'https://api.openai.com/v1'
    resp = httpx.post(f'{base}/chat/completions', json=payload, headers=headers, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    text = data['choices'][0]['message']['content'].strip()
    usage = data.get('usage', {})
    total_tokens = usage.get('total_tokens', 0)
    return text, total_tokens


def generate(prompt_text):
    if USE_API:
        return generate_api(prompt_text)
    else:
        return generate_local(prompt_text)


print('generation functions defined')

generation functions defined


## 10. 执行 pilot run

优化：`no_memory` 条件下同一个 target 的 relevant 和 irrelevant split 共享同一次生成结果。

In [12]:
results = []
no_memory_cache = {}  # target_task_id -> (raw_output, pred_answer, token_usage)

RAW_OUT_DIR = RESULTS_DIR / 'raw_outputs'
RAW_OUT_DIR.mkdir(parents=True, exist_ok=True)

for i, run in enumerate(run_plan):
    tid = run['target_task_id']
    condition = run['condition']
    split = run['split']
    ssid = run['source_set_id']
    atype = run['artifact_type']

    task = all_payload[tid]
    gold = task['answer']

    # Build prompt
    if condition == 'no_memory':
        artifact_content = None
    else:
        artifact_content = artifact_cache[(ssid, atype)]
    prompt = assemble_prompt(task, condition, artifact_content)

    # Generate or dry-run
    raw_output = ''
    pred_answer = ''
    token_usage = 0
    failure_status = 'ok'

    if not RUN_GENERATION:
        failure_status = 'dry_run'
    else:
        # Reuse no_memory result for same target
        if condition == 'no_memory' and tid in no_memory_cache:
            raw_output, pred_answer, token_usage = no_memory_cache[tid]
        else:
            try:
                raw_output, token_usage = generate(prompt)
                pred_answer = extract_answer(raw_output)
                if condition == 'no_memory':
                    no_memory_cache[tid] = (raw_output, pred_answer, token_usage)
            except Exception as e:
                failure_status = f'error: {str(e)[:200]}'
                print(f'  ERROR [{run["run_id"]}]: {e}')

    # Score
    em = compute_em(pred_answer, gold) if pred_answer else 0
    f1 = round(compute_f1(pred_answer, gold), 4) if pred_answer else 0.0

    result = {
        'run_id': run['run_id'],
        'target_task_id': tid,
        'split': split,
        'condition': condition,
        'source_set_id': ssid,
        'routing_decision': 'n/a',
        'memory_attached': 'false' if condition == 'no_memory' else 'true',
        'em': em,
        'f1': f1,
        'token_usage': token_usage,
        'failure_status': failure_status,
        'note': '',
        'pred_answer': pred_answer,
        'gold_answer': gold,
        'prompt_chars': len(prompt),
        'raw_output': raw_output,
        'prompt_text': prompt,
    }
    results.append(result)

    # Write per-run raw output file
    if raw_output or failure_status == 'dry_run':
        raw_path = RAW_OUT_DIR / f"{run['run_id']}.md"
        with raw_path.open('w', encoding='utf-8') as f:
            f.write(f'# {run["run_id"]}\n\n')
            f.write(f'- target_task_id: {tid}\n')
            f.write(f'- split: {split}\n')
            f.write(f'- condition: {condition}\n')
            f.write(f'- source_set_id: {ssid}\n')
            f.write(f'- gold_answer: {gold}\n')
            f.write(f'- pred_answer: {pred_answer}\n')
            f.write(f'- em: {em}\n')
            f.write(f'- f1: {f1}\n')
            f.write(f'- token_usage: {token_usage}\n')
            f.write(f'- prompt_chars: {len(prompt)}\n')
            f.write(f'- failure_status: {failure_status}\n')
            f.write(f'\n---\n\n')
            f.write(f'## Prompt\n\n```\n{prompt}\n```\n\n')
            f.write(f'## Raw Model Output\n\n```\n{raw_output}\n```\n')

    if (i + 1) % 10 == 0 or i == len(run_plan) - 1:
        print(f'progress: {i + 1}/{len(run_plan)}')

print(f'\ntotal results: {len(results)}')
print(f'raw outputs written to: {RAW_OUT_DIR}')


progress: 10/60
progress: 20/60
progress: 30/60
progress: 40/60
progress: 50/60
progress: 60/60

total results: 60
raw outputs written to: /root/2026_SelectTransfer/results/04_pilot_run/raw_outputs


## 11. 写出结果

In [13]:
# ── Write detailed results (with pred/gold/prompt_chars/raw_output) ──
detail_fieldnames = [
    'run_id', 'target_task_id', 'split', 'condition', 'source_set_id',
    'routing_decision', 'memory_attached', 'em', 'f1', 'token_usage',
    'failure_status', 'note', 'pred_answer', 'gold_answer', 'prompt_chars',
    'raw_output',
]
detail_path = RESULTS_DIR / 'pilot_results_detail.csv'
with detail_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=detail_fieldnames)
    writer.writeheader()
    for r in results:
        row = {k: r[k] for k in detail_fieldnames}
        writer.writerow(row)
print(f'detail results -> {detail_path}')

# ── Write pilot_results.csv (matches protocol schema, no pred/gold columns) ──
proto_fieldnames = [
    'run_id', 'target_task_id', 'split', 'condition', 'source_set_id',
    'routing_decision', 'memory_attached', 'em', 'f1', 'token_usage',
    'failure_status', 'note',
]
proto_path = RESULTS_DIR / 'pilot_results.csv'
with proto_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=proto_fieldnames)
    writer.writeheader()
    for r in results:
        row = {k: r[k] for k in proto_fieldnames}
        writer.writerow(row)
print(f'protocol results -> {proto_path}')

# ── Summary of raw output files ──
raw_files = sorted(RAW_OUT_DIR.glob('*.md'))
print(f'\nraw output files: {len(raw_files)}')
total_bytes = sum(f.stat().st_size for f in raw_files)
print(f'total size: {total_bytes / 1024:.1f} KB')


detail results -> /root/2026_SelectTransfer/results/04_pilot_run/pilot_results_detail.csv
protocol results -> /root/2026_SelectTransfer/results/04_pilot_run/pilot_results.csv

raw output files: 60
total size: 414.0 KB


## 12. 快速汇总

In [14]:
def summarize_results(results):
    """Print split x condition aggregated EM and F1."""
    from collections import defaultdict
    buckets = defaultdict(list)
    for r in results:
        key = (r['split'], r['condition'])
        buckets[key].append(r)

    print(f'{"split":<12} {"condition":<32} {"n":>3} {"EM":>6} {"F1":>6}')
    print('-' * 65)
    for split in ['relevant', 'irrelevant']:
        for cond in ['no_memory', 'episodic_trace', 'cross_episode_consolidation']:
            rows = buckets.get((split, cond), [])
            if not rows:
                continue
            n = len(rows)
            avg_em = sum(r['em'] for r in rows) / n
            avg_f1 = sum(r['f1'] for r in rows) / n
            print(f'{split:<12} {cond:<32} {n:>3} {avg_em:>6.2f} {avg_f1:>6.4f}')
        print()


summarize_results(results)

split        condition                          n     EM     F1
-----------------------------------------------------------------
relevant     no_memory                         10   0.30 0.3222
relevant     episodic_trace                    10   0.30 0.3222
relevant     cross_episode_consolidation       10   0.40 0.4222

irrelevant   no_memory                         10   0.30 0.3222
irrelevant   episodic_trace                    10   0.40 0.4222
irrelevant   cross_episode_consolidation       10   0.30 0.3222



## 13. Case-level 检查

逐 case 看 no_memory vs memory 条件的差异，找出最值得保留的 positive / negative transfer case。

In [15]:
def case_comparison(results):
    """Compare no_memory vs memory conditions per target task."""
    from collections import defaultdict
    by_target = defaultdict(list)
    for r in results:
        by_target[r['target_task_id']].append(r)

    for tid in sorted(by_target):
        runs = by_target[tid]
        print(f'\n=== {tid} ===')
        gold = runs[0]['gold_answer']
        print(f'gold: {gold}')
        for r in sorted(runs, key=lambda x: (x['split'], x['condition'])):
            marker = '✓' if r['em'] else '✗'
            print(
                f"  {r['split']:<12} {r['condition']:<32} "
                f"EM={r['em']} F1={r['f1']:.4f} {marker}  "
                f"pred='{r['pred_answer'][:60]}'"
            )


case_comparison(results)


=== wiki_dev_0092 ===
gold: Paris
  irrelevant   cross_episode_consolidation      EM=0 F1=0.0000 ✗  pred='Alex Joffé'
  irrelevant   episodic_trace                   EM=0 F1=0.0000 ✗  pred='Alex Joffé'
  irrelevant   no_memory                        EM=0 F1=0.0000 ✗  pred='Alexandria, Egypt'
  relevant     cross_episode_consolidation      EM=0 F1=0.0000 ✗  pred='Alex Joffé'
  relevant     episodic_trace                   EM=0 F1=0.0000 ✗  pred='Alex Joffé'
  relevant     no_memory                        EM=0 F1=0.0000 ✗  pred='Alexandria, Egypt'

=== wiki_dev_0123 ===
gold: Leave It To Henry
  irrelevant   cross_episode_consolidation      EM=1 F1=1.0000 ✓  pred='Leave It To Henry'
  irrelevant   episodic_trace                   EM=1 F1=1.0000 ✓  pred='Leave It To Henry'
  irrelevant   no_memory                        EM=1 F1=1.0000 ✓  pred='Leave It To Henry'
  relevant     cross_episode_consolidation      EM=1 F1=1.0000 ✓  pred='Leave It To Henry'
  relevant     episodic_trace       

## 14. 清理与下一步

跑完后：

1. 从云端下载 `results/04_pilot_run/` 目录
2. 将 `pilot_results.csv` 复制到本地项目的 `results/pilot_results.csv`
3. 在 `pilot/notes.md` 中记录 Pilot Run Round 1 (Finalized)
4. 按 `protocol/pilot-prompt-scaffold.md` Section 10 的解释对照表分析结果

In [16]:
# Cleanup GPU memory
if model is not None:
    del model
    del tokenizer
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('GPU memory released')

print('\n=== Output files ===')
for p in sorted(RESULTS_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size} bytes)')

GPU memory released

=== Output files ===
  pilot_results.csv  (8006 bytes)
  pilot_results_detail.csv  (11464 bytes)
  raw_outputs  (4096 bytes)
